<br>

<img src="https://www.fedlex.admin.ch/assets/pictos/logo-ch.png" style="width:15%; float:right">

# Tutorial Staatskalender
**Version vom 30.08.2023 - Work in Progress**

Dieses Notebook dient dazu, den Staatskalender in RDF/Linked Data besser zu verstehen, um die Daten kompetent nutzen zu können. Es ist **keine Einführung** - weder in Linked Data noch in SPARQL. Wer die technischen Informationen und Voraussetzungen zu Beginn überspringen möchte, kann direkt zum eigentlichen [Tutorial](#Tutorial) springen.


## Interaktives Notebook

Dieses Tutorial ist ein sogenanntes **interaktives JupyterLite Notebook**. In diesem Notebook können Sie den Inhalt der einzelnen Zellen interaktiv ändern und diese Zellen direkt ausführen, um das Ergebnis Ihrer Änderungen sofort zu sehen. Die Zellen enthalten entweder [Markdown](https://en.wikipedia.org/wiki/Markdown)-Inhalt (wie diese Zelle) oder ausführbaren Python-Quellcode. Dies ist für ein Tutorial sehr hilfreich, weil Inhalte beliebig mit ausführbarem Quellcode kombiniert werden können. Es können also Abfragen gezeigt werden, diese erklärt werden und darauf weiter aufgebaut werden.

**Um direkt loslegen zu können klicken Sie oben im Menu auf Run -> Run All Cells.**  
**Einzelne ausgewählte Zellen können sie danach abändern und mit dem "Play-Button" erneut ausführen und so Abfragen individuell anpassen.**

Das Notebook startet mit einem [Setup](#Setup) der Programierumgebung. Das eigentliche Tutorial startet [hier](#Tutorial).

*Zusätzliche Informationen zu JupyterLite:*  
JupyterLite is ein spezielles Jupyter Notebook mit dem Vorteil, vollständig browserbasiert zu sein, ohne eine aufwändige Backend-Infrastruktur zu benötigen. Der Nachteil ist, dass die erstmalige Ausführung der Zellen einige Zeit in Anspruch nehmen kann, weil eine erhebliche Menge von Daten geladen werden muss. Dass eine Zelle noch in Ausführung ist, ist am `[*]` links neben der Zelle erkennbar. Nach Abschluss der Ausführung erscheint statt eines `*` eine Zahl. Vor der ersten Ausführung ist eine leere Klammer `[ ]` zu sehen. Nachfolgende wiederholte Ausführungen von Zellen werden aufgrund der gespeicherten Daten in Ihrem Browser-Cache viel schneller sein. Wenn Sie Änderungen am Tutorial vornehmen, werden diese Änderungen automatisch lokal im Browser-Cache gespeichert. Bei einem darauffolgenden Öffnen werden wiederum die Änderungen aus dem Browser-Cache geladen. Um wieder zur Ursprüngsversion des Tutorials zurückzukehren, müssen entweder die Browser-Daten gelöscht werden, oder Sie öffnen das Tutorial in einem Chrome-Inkognito Fenster, wo die Änderungen nach Schliessen des Fensters aus dem Cache gelöscht werden.

Wenn Sie mehr über die Handhabung von Jupyter Notebooks wissen wollen, finden Sie hier zwei nützliche Ressourcen:

- [Die JupyterLab Interface](https://jupyterlab.readthedocs.io/en/stable/user/interface.html)
- [Das Jupyter Notebook](https://jupyterlab.readthedocs.io/en/stable/user/notebook.html)

## Setup

Eine SPARQL Abfrage ist nichts anderes als ein POST-Request an den entsprechenden Triple Store Datenbank Server. Um diese Requests und die erhaltenen Antworten einfacher handhaben zu können, enthält dieses Notebook vorbereiteten Python Code, der nachfolgend importiert wird. Zusätzlich wird das `pandas` Modul importiert, welches die Möglichkeit bietet, die tabellarischen Daten der SPARQL Abfragen als [Pandas Dataframes](https://pandas.pydata.org/docs/index.html) weiter zu verarbeiten. 

In [1]:
%pip install -q ipywidgets==8.0.4 ipycytoscape networkx nbformat
import ipycytoscape as cy
import networkx as nx
import pandas as pd
from ext.sparql import query, display_result

# Tutorial

Der Staatskalender ist eine Art "Organigramm" der Bundesbehörden. Da dieses Organigramm einer gewissen Dynamik unterworfen ist (Behörden ändern ihren Namen oder werden mit anderen Behörden zusammengelegt, etc.), ist der Staatskalender in RDF/Linked Data gemäss dem unter https://version.link beschriebenen Datenmodell erstellt. Dieses Datenmodell soll im nachfolgenden Teil erkundet werden.

## Versioned Identity Set

Um alle Behörden in eine gemeinsame Menge zu gruppieren, werden sie gemäss [version.link](https://version.link) mit Hilfe des Attributs [`vl:inVersionedIdentitySet`](https://version.link/#inVersionedIdentitySet) dem Set `<https://ld.admin.ch/ou>` zugeteilt.

## Identitäten und Versionen

Daten gemäss [version.link](https://version.link) werden als Identitäten und Versionen modelliert. Die Identität ist dabei quasi die unabänderliche Existenz einer Behörde und die Versionen sind die zeitlichen Änderungen der Behörden. Jede Identität besteht dabei aus einer Abfolge von mindestens einer Version und die Informationen der Identität beruhen auf der letzten Version.

Anders ausgedrück beschreiben die Identitäten den **Ist-Zustand** und die Versionen die **historische Entwicklung**. Der Ist-Zustand wird dabei auch als Identity-Graph und die historische Entwicklung als Version-Graph bezeichnet.

## Aktuelle Behörden und ihr jeweiliger Name

Um eine Liste aller aktuellen Behörden zu erhalten, muss nach der Klasse [`vl:Identity`](https://version.link/#Identity) gesucht werden und diese Entitäten müssen im entsprechenden Versioned Identity Set sein:

In [2]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX vl: <https://version.link/>

SELECT DISTINCT * WHERE {

    ?org a vl:Identity;
        vl:inVersionedIdentitySet <https://ld.admin.ch/ou>;
        schema:name ?name.
        
    FILTER(lang(?name) = 'de')
}

LIMIT 10

""")

display_result(df)

,org,name
0,https://ld.admin.ch/department/I,Eidgenössisches Departement für auswärtige Angelegenheiten
1,https://ld.admin.ch/department/II,Eidgenössisches Departement des Innern
2,https://ld.admin.ch/department/III,Eidgenössisches Justiz- und Polizeidepartement
3,https://ld.admin.ch/department/IV,"Eidgenössisches Departement für Verteidigung, Bevölkerungsschutz und Sport"
4,https://ld.admin.ch/department/V,Eidgenössisches Finanzdepartement
5,https://ld.admin.ch/department/VI,"Eidgenössisches Departement für Wirtschaft, Bildung und Forschung"
6,https://ld.admin.ch/department/VII,"Eidgenössisches Departement für Umwelt, Verkehr, Energie und Kommunikation"
7,https://ld.admin.ch/office/I.1.1,Generalsekretariat
8,https://ld.admin.ch/office/I.1.2,Staatssekretariat EDA
9,https://ld.admin.ch/office/I.1.4,Direktion für Völkerrecht


## Anzahl aktueller Behörden

Die aktuelle Anzahl aller Behörden im Staatskalender kann folgendermassen ermittelt werden:

In [3]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX vl: <https://version.link/>

SELECT (COUNT(?org) as ?number) WHERE {

    ?org a vl:Identity;
        vl:inVersionedIdentitySet <https://ld.admin.ch/ou>.
        
    FILTER NOT EXISTS {?org a vl:Deprecated.}

}

""")

display_result(df)

,number
0,12895


## Behörden, die so nicht mehr existieren

Behörden, die aufhören, zu existieren, werden nicht einfach gelöscht (auch deren Identität nicht), sondern erhalten zusätzlich die Klasse [`vl:Deprecated`](https://version.link/#Deprecated) zugewiesen:

In [4]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX vl: <https://version.link/>

SELECT * WHERE {

    ?org a vl:Identity;
        vl:inVersionedIdentitySet <https://ld.admin.ch/ou>;
        schema:name ?name;
        a vl:Deprecated.
        
    FILTER(lang(?name) = 'de')
} 

LIMIT 20

""")

display_result(df)

,org,name
0,https://ld.admin.ch/ou/10000805,Konsulat Le Lamentin
1,https://ld.admin.ch/ou/20039877,Berater und FachreferentInnen
2,https://ld.admin.ch/ou/10002715,Sektion Informatik und Services
3,https://ld.admin.ch/ou/10002922,Stab
4,https://ld.admin.ch/ou/10002923,Zivilschutz
5,https://ld.admin.ch/ou/10002927,Informatik
6,https://ld.admin.ch/ou/10002934,Kulturgüterschutz
7,https://ld.admin.ch/ou/10002937,Ausbildung
8,https://ld.admin.ch/ou/10002973,Telematik
9,https://ld.admin.ch/ou/10004256,Mitarbeitende Spezial


## Behörden mit einer "langen" Versionsgeschichte

Folgende Behörden haben eine Versionsgeschichte mit mehr als 5 Versionen:

In [5]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX vl: <https://version.link/>

SELECT ?id ?name ?versions WHERE {
    
    ?id schema:name ?name.
    
    FILTER(lang(?name) = 'de')
    FILTER(?number > 5)
        
    {
    SELECT ?id (GROUP_CONCAT(?version; separator=" ← ") AS ?versions) (COUNT(?version) AS ?number) WHERE {

      ?version a vl:Version;
          vl:identity ?id.

    } 
    GROUP BY ?id
    }
}

ORDER BY DESC(?number)

""")

display_result(df)

,id,name,versions
0,https://ld.admin.ch/ou/manual-oe-297,nHEC,"<a href=""https://ld.admin.ch/ou/manual-oe-297/2023-01-01 ← https://ld.admin.ch/ou/manual-oe-297/2023-04-04 ← https://ld.admin.ch/ou/manual-oe-297/2023-04-05 ← https://ld.admin.ch/ou/manual-oe-297/2023-04-11 ← https://ld.admin.ch/ou/manual-oe-297/2023-04-12 ← https://ld.admin.ch/ou/manual-oe-297/2023-04-13 ← https://ld.admin.ch/ou/manual-oe-297/2023-04-14 ← https://ld.admin.ch/ou/manual-oe-297/2023-04-26 ← https://ld.admin.ch/ou/manual-oe-297/2023-04-27 ← https://ld.admin.ch/ou/manual-oe-297/2023-04-30 ← https://ld.admin.ch/ou/manual-oe-297/2023-05-01 ← https://ld.admin.ch/ou/manual-oe-297/2023-06-09 ← https://ld.admin.ch/ou/manual-oe-297/2023-06-10 ← https://ld.admin.ch/ou/manual-oe-297/2023-06-17 ← https://ld.admin.ch/ou/manual-oe-297/2023-06-18 ← https://ld.admin.ch/ou/manual-oe-297/2023-07-27 ← https://ld.admin.ch/ou/manual-oe-297/2023-07-28 ← https://ld.admin.ch/ou/manual-oe-297/2023-08-03 ← https://ld.admin.ch/ou/manual-oe-297/2023-08-05"" target=""_blank"">https://ld.admin.ch/ou/manual-oe-297/2023-01-01 ← https://ld.admin.ch/ou/manual-oe-297/2023-04-04 ← https://ld.admin.ch/ou/manual-oe-297/2023-04-05 ← https://ld.admin.ch/ou/manual-oe-297/2023-04-11 ← https://ld.admin.ch/ou/manual-oe-297/2023-04-12 ← https://ld.admin.ch/ou/manual-oe-297/2023-04-13 ← https://ld.admin.ch/ou/manual-oe-297/2023-04-14 ← https://ld.admin.ch/ou/manual-oe-297/2023-04-26 ← https://ld.admin.ch/ou/manual-oe-297/2023-04-27 ← https://ld.admin.ch/ou/manual-oe-297/2023-04-30 ← https://ld.admin.ch/ou/manual-oe-297/2023-05-01 ← https://ld.admin.ch/ou/manual-oe-297/2023-06-09 ← https://ld.admin.ch/ou/manual-oe-297/2023-06-10 ← https://ld.admin.ch/ou/manual-oe-297/2023-06-17 ← https://ld.admin.ch/ou/manual-oe-297/2023-06-18 ← https://ld.admin.ch/ou/manual-oe-297/2023-07-27 ← https://ld.admin.ch/ou/manual-oe-297/2023-07-28 ← https://ld.admin.ch/ou/manual-oe-297/2023-08-03 ← https://ld.admin.ch/ou/manual-oe-297/2023-08-05"
1,https://ld.admin.ch/ou/20017377,European Free Trade Ass. EFTA Geneva,"<a href=""https://ld.admin.ch/ou/20017377/2023-01-01 ← https://ld.admin.ch/ou/20017377/2023-07-16 ← https://ld.admin.ch/ou/20017377/2023-07-19 ← https://ld.admin.ch/ou/20017377/2023-07-22 ← https://ld.admin.ch/ou/20017377/2023-07-24 ← https://ld.admin.ch/ou/20017377/2023-07-26 ← https://ld.admin.ch/ou/20017377/2023-08-01 ← https://ld.admin.ch/ou/20017377/2023-08-03 ← https://ld.admin.ch/ou/20017377/2023-08-06 ← https://ld.admin.ch/ou/20017377/2023-08-09 ← https://ld.admin.ch/ou/20017377/2023-08-12 ← https://ld.admin.ch/ou/20017377/2023-08-14 ← https://ld.admin.ch/ou/20017377/2023-08-17 ← https://ld.admin.ch/ou/20017377/2023-08-20 ← https://ld.admin.ch/ou/20017377/2023-08-26 ← https://ld.admin.ch/ou/20017377/2023-08-28"" target=""_blank"">https://ld.admin.ch/ou/20017377/2023-01-01 ← https://ld.admin.ch/ou/20017377/2023-07-16 ← https://ld.admin.ch/ou/20017377/2023-07-19 ← https://ld.admin.ch/ou/20017377/2023-07-22 ← https://ld.admin.ch/ou/20017377/2023-07-24 ← https://ld.admin.ch/ou/20017377/2023-07-26 ← https://ld.admin.ch/ou/20017377/2023-08-01 ← https://ld.admin.ch/ou/20017377/2023-08-03 ← https://ld.admin.ch/ou/20017377/2023-08-06 ← https://ld.admin.ch/ou/20017377/2023-08-09 ← https://ld.admin.ch/ou/20017377/2023-08-12 ← https://ld.admin.ch/ou/20017377/2023-08-14 ← https://ld.admin.ch/ou/20017377/2023-08-17 ← https://ld.admin.ch/ou/20017377/2023-08-20 ← https://ld.admin.ch/ou/20017377/2023-08-26 ← https://ld.admin.ch/ou/20017377/2023-08-28"
2,https://ld.admin.ch/ou/20017378,SECO MA unbezahlter Urlaub,"<a href=""https://ld.admin.ch/ou/20017378/2023-01-01 ← https://ld.admin.ch/ou/20017378/2023-07-16 ← https://ld.admin.ch/ou/20017378/2023-07-19 ← https://ld.admin.ch/ou/20017378/2023-07-22 ← https://ld.admin.ch/ou/20017378/2023-07-24 ← https://ld.admin.ch/ou/20017378/2023-07-26 ← https://ld.admin.ch/ou/20017378/2023-08-01 ← https://ld.admin.ch/ou/20017378/2023-08-03 ← https://ld.admin.ch/ou/20017378/2023-08

## "Wiederauferstandene" Behörden

Die nachfolgenden Versionen haben vl:successor mit schema:startDate > als schema:endDate der Version, es existiert also eine zeitliche Lücke, in der die Behörde "pausiert" hat.

In [6]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX vl: <https://version.link/>

SELECT * WHERE {

?version vl:successor ?succ;
    vl:inVersionedIdentitySet <https://ld.admin.ch/ou>;
    schema:endDate ?endDate.

?succ schema:startDate ?startDate.

BIND(day(?startDate - ?endDate) as ?diff)
FILTER(?diff > 0)

}
ORDER BY DESC(?diff)

LIMIT 10

""")

display_result(df)

,version,succ,endDate,startDate,diff
0,https://ld.admin.ch/ou/609015944/2023-01-01,https://ld.admin.ch/ou/609015944/2023-08-05,2023-01-14,2023-08-05,203
1,https://ld.admin.ch/ou/1109697853/2023-02-03,https://ld.admin.ch/ou/1109697853/2023-07-05,2023-04-12,2023-07-05,84
2,https://ld.admin.ch/ou/272791558/2023-01-01,https://ld.admin.ch/ou/272791558/2023-07-02,2023-04-15,2023-07-02,78
3,https://ld.admin.ch/ou/2017922218/2023-04-05,https://ld.admin.ch/ou/2017922218/2023-08-10,2023-06-03,2023-08-10,68
4,https://ld.admin.ch/ou/1108839157/2023-01-01,https://ld.admin.ch/ou/1108839157/2023-08-10,2023-06-04,2023-08-10,67
5,https://ld.admin.ch/ou/267629368/2023-01-01,https://ld.admin.ch/ou/267629368/2023-06-14,2023-06-03,2023-06-14,11
6,https://ld.admin.ch/ou/20037340/2023-01-01,https://ld.admin.ch/ou/20037340/2023-06-10,2023-06-01,2023-06-10,9
7,https://ld.admin.ch/ou/20037335/2023-01-01,https://ld.admin.ch/ou/20037335/2023-06-10,2023-06-01,2023-06-10,9
8,https://ld.admin.ch/ou/20047851/2023-01-01,https://ld.admin.ch/ou/20047851/2023-01-16,2023-01-08,2023-01-16,8
9,https://ld.admin.ch/ou/1722782359/2023-06-24,https://ld.admin.ch/ou/1722782359/2023-07-03,2023-06-26,2023-07-03,7


## Behörden mit gleichen Namen

Folgendes Beispiel zeigt diejenigen Behörden, von denen mehrere mit gleichem Namen (an anderer Stelle im Organigramm) existieren. Dieses Beispiel zeigt, wie die von der SPARQL Query gelieferte Daten mit dem gesamten Spektrum der Datenverarbeitungsmöglichkeit von Python weiterverarbeitet werden kann:

In [7]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX vl: <https://version.link/>

SELECT * WHERE {

    ?identity a vl:Identity;
        schema:name ?name.
    
    FILTER(lang(?name) = 'de')

    FILTER NOT EXISTS {?identity a vl:Deprecated.}

}

ORDER BY ?name

""")

df['non_unique'] = df.duplicated(subset = 'name', keep = False)

df = df[df['non_unique'] == True]

display_result(df[["identity", "name"]].head(100))

,identity,name
165,https://ld.admin.ch/ou/968521976,ALK Team A
166,https://ld.admin.ch/ou/649483307,ALK Team A
167,https://ld.admin.ch/ou/392902542,ALK Team B
168,https://ld.admin.ch/ou/140974931,ALK Team B
169,https://ld.admin.ch/ou/1199404241,ALK Team C
170,https://ld.admin.ch/ou/1483823116,ALK Team C
226,https://ld.admin.ch/ou/1373859324,AVOR
227,https://ld.admin.ch/ou/60031960-11-05,AVOR
349,https://ld.admin.ch/ou/10002716,Abteilung Internationales
350,https://ld.admin.ch/ou/10004554,Abteilung Internationales


## Behörden und ihr Pfad im Organigramm

Nachfolgende Query ermittelt alle Sub-Behörden einer frei wählbaren Behörde und zeigt den "Pfad" der entsprechenden Sub-Behörde an:

In [8]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX vl: <https://version.link/>

SELECT (GROUP_CONCAT(CONCAT(STR(?level), ":", STR(?parent_name)); separator= " → ") AS ?hierarchy) WHERE {

    BIND ('de' AS ?lang)

    ?parent schema:subOrganization* ?org.
    ?parent schema:name ?parent_name.
    FILTER(lang(?parent_name) = ?lang)

    {
    SELECT ?parent (COUNT(?org) AS ?level) WHERE {

        BIND (<https://ld.admin.ch/office/II.1.4> AS ?root) 
        
        #start root, e.g. <https://ld.admin.ch/FCh> = Bundeskanzlei
        #                 <https://ld.admin.ch/office/II.1.4> = Bundesarchiv

        ?root schema:subOrganization+ ?parent.
        ?org schema:subOrganization+ ?parent.


    } GROUP BY ?parent ORDER BY ?level
    }
} GROUP BY ?org ORDER BY ?hierarchy

""")

display_result(df)

,hierarchy
0,5:Direktion
1,5:Direktion → 6:Abteilung Informationszugang
2,5:Direktion → 6:Abteilung Informationszugang → 7:Dienst Digitalisierung
3,5:Direktion → 6:Abteilung Informationszugang → 7:Dienst Informationsbereitstellung
4,5:Direktion → 6:Abteilung Informationszugang → 7:Dienst Informationsnutzung
5,5:Direktion → 6:Abteilung Informationsüberlieferung
6,5:Direktion → 6:Abteilung Informationsüberlieferung → 7:Dienst Datenarchivierung
7,5:Direktion → 6:Abteilung Informationsüberlieferung → 7:Dienst Informationsmanagement
8,5:Direktion → 6:Abteilung Informationsüberlieferung → 7:Dienst Informationsübernahme
9,5:Direktion → 6:Ressort Informationstechnik


## Behörden, die neu gegründet wurden

Folgende Query ermittelt die Behörden, die nach dem 1.1.2023 gegründet wurden:

In [9]:
df = await query("""

PREFIX schema: <http://schema.org/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX vl: <https://version.link/>

SELECT ?org ?name ?date WHERE {

    ?org a vl:Identity;
        vl:version ?version;
        schema:name ?name.
        
    FILTER(lang(?name) = "de")
        
    OPTIONAL {?version vl:predecessor ?predecessor.}
    ?version schema:startDate ?date.
    
    FILTER(!BOUND(?predecessor))
    FILTER(?date > "2023-01-01")

} LIMIT 50

""")

display_result(df)

,org,name,date
0,https://ld.admin.ch/office/II.2.2.4,Ausgleichsfonds AHV/IV/EO,2023-07-21
1,https://ld.admin.ch/office/II.2.2.4,Ausgleichsfonds AHV/IV/EO,2023-07-21
2,https://ld.admin.ch/office/III.2.2.1,Schweizerisches Institut für Rechtsvergleichung,2023-08-10
3,https://ld.admin.ch/office/III.2.2.1,Schweizerisches Institut für Rechtsvergleichung,2023-08-10
4,https://ld.admin.ch/office/III.2.2.3,Eidgenössische Revisionsaufsichtsbehörde,2023-08-10
5,https://ld.admin.ch/office/III.2.2.3,Eidgenössische Revisionsaufsichtsbehörde,2023-08-10
6,https://ld.admin.ch/office/IV.2.1.1,Unabhängige Aufsichtsbehörde über die nachrichtendienstlichen Tätigkeiten,2023-08-10
7,https://ld.admin.ch/office/IV.2.1.1,Unabhängige Aufsichtsbehörde über die nachrichtendienstlichen Tätigkeiten,2023-08-10
8,https://ld.admin.ch/office/V.2.2.2,Pensionskasse des Bundes,2023-07-21
9,https://ld.admin.ch/office/V.2.2.2,Pensionskasse des Bundes,2023-07-21


## Grafisches Organigramm

In [10]:
edge_df = await query("""

PREFIX vl: <https://version.link/>
PREFIX admin: <https://schema.ld.admin.ch/>
PREFIX schema: <http://schema.org/>

SELECT DISTINCT ?source ?sourceName ?target ?targetName
WHERE {

    BIND (<https://ld.admin.ch/office/VII.1.4> AS ?top)
   
    ?source schema:parentOrganization+ ?top;
        a vl:Identity;
        schema:parentOrganization ?target;
        schema:name ?sourceName.
        
    ?target schema:name ?targetName.
    
    FILTER(lang(?sourceName) = "de")
    FILTER(lang(?targetName) = "de")
}


""")

display_result(edge_df)

ids = pd.Series.tolist(edge_df["source"]) + pd.Series.tolist(edge_df["target"])
names = pd.Series.tolist(edge_df["sourceName"]) + pd.Series.tolist(edge_df["targetName"])

node_df = pd.DataFrame(list(zip(ids, names)), columns = ["id", "name"])
node_df.drop_duplicates(inplace=True)


G = nx.from_pandas_edgelist(edge_df, source="source", target="target", create_using=nx.DiGraph())
node_attributes = node_df.set_index('id').to_dict("index")
nx.set_node_attributes(G, node_attributes)


my_style = [
    {
        'selector': 'node',
         'style': {
            'font-family': 'helvetica',
            'font-size': '8px',
            'text-valign': "center",
             'text-wrap': 'wrap',
             'text-max-width': '50px',
             'width': '100px',
             'shape':'roundrectangle',
            'label': 'data(name)' #Zugriff auf das Attribut 'name' der Nodes
         }
    },
    {
        "selector": "edge.directed",
        "style": {
            "curve-style": "bezier",
            "target-arrow-shape": "triangle",
        }
    }
]

orga = cy.CytoscapeWidget()
orga.graph.add_graph_from_networkx(G, directed=True)
orga.set_style(my_style)
orga.set_layout(name = "dagre", rankDir = "BT")
orga

,source,sourceName,target,targetName
0,https://ld.admin.ch/ou/10002556,Geschäftsleitung,https://ld.admin.ch/office/VII.1.4,Bundesamt für Energie
1,https://ld.admin.ch/ou/20028326,Medien und Politik,https://ld.admin.ch/ou/10002556,Geschäftsleitung
2,https://ld.admin.ch/ou/10002557,Publishing,https://ld.admin.ch/ou/20028326,Medien und Politik
3,https://ld.admin.ch/ou/10002558,Betriebswirtschaft und Organisation,https://ld.admin.ch/ou/10002556,Geschäftsleitung
4,https://ld.admin.ch/ou/10002559,Finanzen und Controlling,https://ld.admin.ch/ou/10002558,Betriebswirtschaft und Organisation
5,https://ld.admin.ch/ou/10002560,Informatik,https://ld.admin.ch/ou/10002558,Betriebswirtschaft und Organisation
6,https://ld.admin.ch/ou/10002561,Energiewirtschaft,https://ld.admin.ch/ou/10002556,Geschäftsleitung
7,https://ld.admin.ch/ou/10002566,Energieeffizienz / erneuerbare Energien,https://ld.admin.ch/ou/10002556,Geschäftsleitung
8,https://ld.admin.ch/ou/10002569,Gebäude,https://ld.admin.ch/ou/10002566,Energieeffizienz / erneuerbare Energien
9,https://ld.admin.ch/ou/10002570,Erneuerbare Energien,https://ld.admin.ch/ou/10002566,Energieeffizienz / erneuerbare Energien


CytoscapeWidget(cytoscape_layout={'name': 'dagre', 'rankDir': 'BT'}, cytoscape_style=[{'selector': 'node', 'st…